# DrugResponse — predicting pCR to neoadjuvant chemotherapy in breast cancer

**Question.** Can tumour gene expression measured *before* treatment predict pathological complete response (**pCR**) to neoadjuvant chemotherapy better than standard clinical features alone?

**Data.** Two public GEO cohorts, same microarray platform (Affymetrix HG-U133A, GPL96):
- **GSE25065** (MD Anderson, taxane → anthracycline)
- **GSE20194** (MAQC-II, paclitaxel + FAC)

**Approach.**
1. Parse GEO series matrices → clinical table, target `y`, expression matrix
2. Harmonise clinical features across cohorts; per-cohort gene standardisation
3. Three models: **A** clinical only (logistic regression), **B** clinical + genes (logistic regression), **C** clinical + genes (XGBoost)
4. **Nested cross-validation**: outer loop estimates performance, inner loop tunes hyperparameters; gene selection happens *inside* each training fold (no leakage)
5. ROC-AUC with bootstrap 95% CI, subgroup analysis, SHAP interpretation, limitations

Run everything: **Runtime → Run all** (≈ 5–10 min on the free CPU runtime).

## 0. Setup

In [ ]:
import gzip, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score, brier_score_loss
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 50)

In [ ]:
# Download data from GEO (Colab has internet). Re-run after a runtime reset.
!mkdir -p data/raw
!wget -q -nc -P data/raw https://ftp.ncbi.nlm.nih.gov/geo/series/GSE25nnn/GSE25065/matrix/GSE25065_series_matrix.txt.gz
!wget -q -nc -P data/raw https://ftp.ncbi.nlm.nih.gov/geo/series/GSE20nnn/GSE20194/matrix/GSE20194_series_matrix.txt.gz
!wget -q -nc -P data/raw https://ftp.ncbi.nlm.nih.gov/geo/platforms/GPLnnn/GPL96/annot/GPL96.annot.gz
!ls -lh data/raw

## 1. Parse GEO series matrices

**Logic.** A series matrix file has two parts:
- metadata lines starting with `!` — patient characteristics are in `!Sample_characteristics_ch1` lines, each cell formatted `"key: value"`; patient IDs are in `!Sample_geo_accession`;
- the expression table between `!series_matrix_table_begin` and `!series_matrix_table_end` — rows are probes, columns are patients.

⚠️ **Parsing trap.** A characteristics line is *not* guaranteed to hold the same key for every patient. In GSE20194, samples that lack a field (e.g. `race`) have all later fields shifted up one line. Taking the key from the first cell of each line silently mis-assigns fields for 20 patients and drops 10 labelled patients. We therefore read the key from **every cell**.

We return `clinical` (patients × characteristics) and `expr` (patients × probes), so that rows are always patients, as scikit-learn expects.

In [ ]:
def parse_series_matrix(path):
    """Return (clinical, expr): both DataFrames indexed by GSM patient ID."""
    sample_ids, char_lines, table_lines = None, [], []
    in_table = False
    with gzip.open(path, "rt", encoding="latin-1") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith("!series_matrix_table_begin"):
                in_table = True
                continue
            if line.startswith("!series_matrix_table_end"):
                break
            if in_table:
                table_lines.append(line)
            elif line.startswith("!Sample_geo_accession"):
                sample_ids = [x.strip('"') for x in line.split("\t")[1:]]
            elif line.startswith("!Sample_characteristics_ch1"):
                char_lines.append([x.strip('"') for x in line.split("\t")[1:]])

    # Parse every cell as its own "key: value" pair. Do NOT assume one key per line:
    # in GSE20194 some samples lack a field, so their values are shifted to other lines.
    records = {sid: {} for sid in sample_ids}
    for values in char_lines:
        assert len(values) == len(sample_ids)
        for sid, cell in zip(sample_ids, values):
            if ": " in cell:
                key, value = cell.split(": ", 1)
                assert key.strip() not in records[sid], f"duplicate key {key!r} for {sid}"
                records[sid][key.strip()] = value.strip()

    clinical = pd.DataFrame.from_dict(records, orient="index").reindex(sample_ids)
    clinical = clinical.replace({"NA": np.nan, "": np.nan})

    from io import StringIO
    expr = pd.read_csv(StringIO("\n".join(table_lines)), sep="\t", index_col=0)
    expr.columns = [c.strip('"') for c in expr.columns]
    expr.index = [str(i).strip('"') for i in expr.index]
    expr = expr.T.loc[sample_ids].astype(float)   # patients x probes
    return clinical, expr


clin_25065, expr_25065 = parse_series_matrix("data/raw/GSE25065_series_matrix.txt.gz")
clin_20194, expr_20194 = parse_series_matrix("data/raw/GSE20194_series_matrix.txt.gz")

print("GSE25065:", clin_25065.shape, expr_25065.shape)
print("GSE20194:", clin_20194.shape, expr_20194.shape)
clin_25065.head(3)

## 2. Target and harmonised clinical features

**Logic.** The two cohorts name the same things differently (`er_status_ihc` vs `er_status`, `clinical_t_stage: T2` vs `tbefore: 2`). We map both onto one schema:

| feature | meaning |
|---|---|
| `age` | age at diagnosis |
| `er`, `pr`, `her2` | receptor status, 1 = positive |
| `t_stage` | tumour size category (1–4) |
| `n_stage` | lymph-node category (0–3) |
| `grade` | histological grade (1–3) |

Target: `y = 1` if pCR, `0` if residual disease (RD).

⚠️ **Leakage trap.** GSE25065 also contains `dlda30_prediction`, `rcb_0_i_prediction`, `chemosensitivity_prediction` (other models' predictions of pCR) and `pathologic_response_rcb_class` (measured *after* surgery). We never use them as features — they would give a near-perfect but fake AUC.

In [ ]:
def to_num(s):
    return pd.to_numeric(s, errors="coerce")

def pos_neg(s):
    return s.map({"P": 1, "N": 0, "Positive": 1, "Negative": 0})

def stage_num(s):
    # "T2" -> 2, "N1" -> 1, "2" -> 2; anything else -> NaN
    return to_num(s.astype(str).str.extract(r"(\d)")[0])


def harmonise_25065(c):
    df = pd.DataFrame(index=c.index)
    df["age"] = to_num(c["age_years"])
    df["er"] = pos_neg(c["er_status_ihc"])
    df["pr"] = pos_neg(c["pr_status_ihc"])
    df["her2"] = pos_neg(c["her2_status"])
    df["t_stage"] = stage_num(c["clinical_t_stage"])
    df["n_stage"] = stage_num(c["clinical_nodal_status"])
    df["grade"] = to_num(c["grade"])
    y = c["pathologic_response_pcr_rd"].map({"pCR": 1, "RD": 0})
    return df, y


def harmonise_20194(c):
    df = pd.DataFrame(index=c.index)
    df["age"] = to_num(c["age"])
    df["er"] = pos_neg(c["er_status"])
    df["pr"] = pos_neg(c["pr_status"])
    df["her2"] = pos_neg(c["her2 status"])
    df["t_stage"] = stage_num(c["tbefore"])
    df["n_stage"] = stage_num(c["nbefore"])
    df["grade"] = to_num(c["bmngrd"])
    y = c["pcr_vs_rd"].map({"pCR": 1, "RD": 0})
    return df, y


X_clin_a, y_a = harmonise_25065(clin_25065)
X_clin_b, y_b = harmonise_20194(clin_20194)

for name, Xc, yy in [("GSE25065", X_clin_a, y_a), ("GSE20194", X_clin_b, y_b)]:
    print(f"{name}: n={len(yy)}, pCR rate={yy.mean():.3f}, missing labels={yy.isna().sum()}")
X_clin_a.describe().round(2)

## 3. Gene expression: per-cohort standardisation and pooling

**Logic.**
- Both cohorts are already log2-scaled and on the same platform, so probes match one-to-one.
- Different labs → systematic shifts between cohorts (**batch effect**). A simple, label-free fix: **z-score every probe within each cohort** (mean 0, sd 1). It uses no outcome information, so it does not leak the target.
- We also add a `cohort` indicator so the clinical model can absorb remaining baseline differences.
- Patients without a pCR/RD label are dropped.

In [ ]:
def zscore_within_cohort(expr):
    return (expr - expr.mean()) / expr.std(ddof=0).replace(0, 1)

common_probes = expr_25065.columns.intersection(expr_20194.columns)
# drop Affymetrix control probes (AFFX-...)
common_probes = [p for p in common_probes if not p.startswith("AFFX")]

expr = pd.concat([
    zscore_within_cohort(expr_25065[common_probes]),
    zscore_within_cohort(expr_20194[common_probes]),
])
clinical = pd.concat([X_clin_a.assign(cohort=0), X_clin_b.assign(cohort=1)])
y = pd.concat([y_a, y_b])

keep = y.notna()
expr, clinical, y = expr[keep], clinical[keep], y[keep].astype(int)

CLINICAL_COLS = list(clinical.columns)
GENE_COLS = list(expr.columns)
X = pd.concat([clinical, expr], axis=1)

assert (X.index == y.index).all()
print(f"Pooled cohort: {X.shape[0]} patients, {len(CLINICAL_COLS)} clinical + {len(GENE_COLS)} probe features")
print(f"pCR rate: {y.mean():.3f}")

## 4. Models and leakage-safe pipelines

**Logic.**
- With ~22,000 probes and ~460 patients, we must reduce features. We keep the **top-k most variable probes**, but this choice is made **inside the pipeline**, so it is re-computed on the training part of every fold and never sees the test patients.
- Clinical columns: impute missing values with the median (also learned on training data only), then scale.
- Three models:
  - **A — clinical only**, logistic regression (the baseline a clinician already has)
  - **B — clinical + genes**, L2-regularised logistic regression
  - **C — clinical + genes**, XGBoost (gradient-boosted trees, can capture non-linear effects)
- Class imbalance (~20% pCR): `class_weight="balanced"` / `scale_pos_weight` so the model does not just predict "no pCR" for everyone.

In [ ]:
class TopVarianceSelector(BaseEstimator, TransformerMixin):
    """Keep the k columns with the highest variance — computed on training data only."""
    def __init__(self, k=500):
        self.k = k

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        k = min(self.k, X.shape[1])
        self.idx_ = np.argsort(np.nanvar(X, axis=0))[::-1][:k]
        return self

    def transform(self, X):
        return np.asarray(X, dtype=float)[:, self.idx_]


clinical_prep = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
gene_prep = Pipeline([("select", TopVarianceSelector(k=500)), ("scale", StandardScaler())])

pos_weight = (y == 0).sum() / (y == 1).sum()

MODELS = {
    "A_clinical_LR": (
        Pipeline([
            ("prep", ColumnTransformer([("clin", clinical_prep, CLINICAL_COLS)])),
            ("clf", LogisticRegression(class_weight="balanced", max_iter=2000)),
        ]),
        {"clf__C": [0.01, 0.1, 1.0]},
    ),
    "B_clinical_genes_LR": (
        Pipeline([
            ("prep", ColumnTransformer([("clin", clinical_prep, CLINICAL_COLS),
                                        ("genes", gene_prep, GENE_COLS)])),
            ("clf", LogisticRegression(class_weight="balanced", max_iter=5000)),
        ]),
        {"prep__genes__select__k": [200, 500], "clf__C": [0.001, 0.01, 0.1]},
    ),
    "C_clinical_genes_XGB": (
        Pipeline([
            ("prep", ColumnTransformer([("clin", clinical_prep, CLINICAL_COLS),
                                        ("genes", gene_prep, GENE_COLS)])),
            ("clf", XGBClassifier(n_estimators=300, learning_rate=0.05, subsample=0.8,
                                  colsample_bytree=0.5, scale_pos_weight=pos_weight,
                                  eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1)),
        ]),
        {"prep__genes__select__k": [200, 500], "clf__max_depth": [2, 3]},
    ),
}
list(MODELS)

## 5. Nested cross-validation

**Logic.**
- **Outer loop** (5 stratified folds): each patient is in the test fold exactly once → we get an honest, out-of-fold (OOF) prediction for every patient.
- **Inner loop** (3 stratified folds, `GridSearchCV`): runs *only on the outer-training part* to choose hyperparameters.
- If we tuned and evaluated on the same folds, we would pick the setting that happens to fit those test patients best, and the AUC would be optimistic.
- Metric: **ROC-AUC** (how well the model ranks pCR above non-pCR patients); we also report **PR-AUC** (more informative with imbalance; random = pCR rate) and **Brier score** (calibration, lower is better).

In [ ]:
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

oof = {}           # model name -> out-of-fold probabilities
chosen_params = {} # model name -> best params per outer fold

for name, (pipe, grid) in MODELS.items():
    probs = pd.Series(np.nan, index=y.index)
    chosen_params[name] = []
    for fold, (tr, te) in enumerate(outer_cv.split(X, y)):
        search = GridSearchCV(pipe, grid, scoring="roc_auc", cv=inner_cv, n_jobs=-1)
        search.fit(X.iloc[tr], y.iloc[tr])
        probs.iloc[te] = search.predict_proba(X.iloc[te])[:, 1]
        chosen_params[name].append(search.best_params_)
    oof[name] = probs
    print(f"{name:22s} outer-fold AUC = {roc_auc_score(y, probs):.3f}")

## 6. Results with bootstrap 95% confidence intervals

**Logic.** One AUC number hides uncertainty. We resample patients with replacement 1,000 times, recompute AUC each time, and take the 2.5th–97.5th percentiles. We also bootstrap the **difference** B − A on the same resamples: if its CI includes 0, genes do not give a clearly significant gain.

In [ ]:
def bootstrap_auc(y_true, p, n_boot=1000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    y_true, p = np.asarray(y_true), np.asarray(p)
    aucs = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y_true), len(y_true))
        if y_true[i].min() == y_true[i].max():
            continue
        aucs.append(roc_auc_score(y_true[i], p[i]))
    return np.percentile(aucs, [2.5, 97.5])

rows = []
for name, p in oof.items():
    lo, hi = bootstrap_auc(y, p)
    rows.append({"model": name,
                 "ROC-AUC": roc_auc_score(y, p), "95% CI": f"{lo:.3f}–{hi:.3f}",
                 "PR-AUC": average_precision_score(y, p),
                 "Brier": brier_score_loss(y, p)})
results = pd.DataFrame(rows).set_index("model").round(3)
print(f"PR-AUC of a random model = pCR rate = {y.mean():.3f}")
results

In [ ]:
def bootstrap_delta(y_true, p1, p0, n_boot=1000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    y_true, p1, p0 = map(np.asarray, (y_true, p1, p0))
    deltas = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y_true), len(y_true))
        if y_true[i].min() == y_true[i].max():
            continue
        deltas.append(roc_auc_score(y_true[i], p1[i]) - roc_auc_score(y_true[i], p0[i]))
    return np.mean(deltas), np.percentile(deltas, [2.5, 97.5])

for name in ["B_clinical_genes_LR", "C_clinical_genes_XGB"]:
    d, (lo, hi) = bootstrap_delta(y, oof[name], oof["A_clinical_LR"])
    print(f"{name} − A_clinical_LR: ΔAUC = {d:+.3f} (95% CI {lo:+.3f} to {hi:+.3f})")

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
for name, p in oof.items():
    fpr, tpr, _ = roc_curve(y, p)
    ax.plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(y, p):.3f})")
ax.plot([0, 1], [0, 1], "--", color="grey", lw=1)
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title("Out-of-fold ROC curves (nested CV)")
ax.legend(fontsize=8, loc="lower right")
os.makedirs("figures", exist_ok=True)
plt.tight_layout(); plt.savefig("figures/roc_curves.png", dpi=150); plt.show()

## 7. Subgroup analysis

**Logic.** Breast cancer is several diseases. pCR rates and biology differ a lot between **HR+/HER2−**, **HER2+** and **triple-negative (TNBC)**. Part of the overall AUC may simply come from recognising the subtype. Within-subgroup AUC shows whether the model ranks patients *inside* each subtype.

In [ ]:
hr_pos = (clinical["er"] == 1) | (clinical["pr"] == 1)
subtype = pd.Series("unknown", index=y.index)
subtype[(clinical["her2"] == 1)] = "HER2+"
subtype[(clinical["her2"] == 0) & hr_pos] = "HR+/HER2-"
subtype[(clinical["her2"] == 0) & (clinical["er"] == 0) & (clinical["pr"] == 0)] = "TNBC"

rows = []
for g in ["HR+/HER2-", "HER2+", "TNBC"]:
    m = subtype == g
    row = {"subtype": g, "n": int(m.sum()), "pCR rate": round(y[m].mean(), 3)}
    for name, p in oof.items():
        row[name] = round(roc_auc_score(y[m], p[m]), 3) if y[m].nunique() == 2 else np.nan
    rows.append(row)
pd.DataFrame(rows).set_index("subtype")

## 8. Final model and interpretation (SHAP)

**Logic.** Nested CV *estimates* performance; for interpretation we refit the XGBoost pipeline on **all** patients with the most frequently chosen hyperparameters. **SHAP** values show how much each feature pushes an individual prediction up or down; the mean |SHAP| ranks features globally. Probe IDs are mapped to gene symbols with the GPL96 annotation.

In [ ]:
from collections import Counter
import shap

name = "C_clinical_genes_XGB"
pipe, _ = MODELS[name]
best = Counter(tuple(sorted(d.items())) for d in chosen_params[name]).most_common(1)[0][0]
final = pipe.set_params(**dict(best)).fit(X, y)
print("final hyperparameters:", dict(best))

# feature names after preprocessing
k = final.named_steps["prep"].named_transformers_["genes"].named_steps["select"].idx_
feature_names = CLINICAL_COLS + [GENE_COLS[i] for i in k]

# probe -> gene symbol (GPL96 annotation from GEO); fall back to probe IDs if unavailable
probe2gene = {}
try:
    with gzip.open("data/raw/GPL96.annot.gz", "rt", encoding="latin-1") as f:
        lines = f.read().split("\n")
    start = lines.index("!platform_table_begin") + 1
    end = lines.index("!platform_table_end")
    from io import StringIO
    annot = pd.read_csv(StringIO("\n".join(lines[start:end])), sep="\t", dtype=str)
    probe2gene = dict(zip(annot["ID"], annot["Gene symbol"]))
except Exception as e:
    print("annotation not available, showing probe IDs:", e)

pretty = [f"{probe2gene.get(f, f)} ({f})" if f in probe2gene else f for f in feature_names]

Xt = final.named_steps["prep"].transform(X)
explainer = shap.TreeExplainer(final.named_steps["clf"])
shap_values = explainer.shap_values(Xt)
shap.summary_plot(shap_values, Xt, feature_names=pretty, max_display=15, show=False)
plt.tight_layout(); plt.savefig("figures/shap_summary.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
importance = (pd.DataFrame({"feature": pretty, "mean_abs_shap": np.abs(shap_values).mean(axis=0)})
              .sort_values("mean_abs_shap", ascending=False).head(15).reset_index(drop=True))
importance.round(4)

## 9. Conclusions and limitations

**Fill in after running** (numbers from sections 6–7):
- Clinical-only baseline ROC-AUC: …
- Adding gene expression changes ROC-AUC by … (95% CI …) → genes add *clear / modest / no clear* value.
- Subtype: performance is highest in … and lowest in …

**Limitations**
- ~460 patients from two US cohorts, older microarray platform; modern trials use RNA-seq.
- No truly external test set: both cohorts are pooled and evaluated with cross-validation. Next step: train on one cohort, test on the other (or on I-SPY2).
- Per-cohort z-scoring is a simple batch correction; it assumes similar case mix in both cohorts.
- Chemotherapy regimens (taxane + anthracycline) differ from current standards with immunotherapy / anti-HER2 therapy.
- SHAP explains the *model*, not biology: top genes are hypotheses, not causal markers.